# Week 11 — Testing, Debugging, and Performance
## LeafGuard AI: Building Confidence Before Release

A project is not ready just because features exist. It must also be testable, debuggable, and fast enough for users. This notebook explores those ideas with Python.

**Learning outcomes:**
- Write unit tests with `unittest`
- Measure latency with `time.perf_counter`
- Compute summary statistics for performance data
- Parse logs to discover failures
- Generate edge cases for robust testing

---


## Cell 1: Define a Small Function and Unit Tests

Unit tests should isolate one behavior at a time.


In [ ]:
import unittest

def format_confidence(score):
    if not 0.0 <= score <= 1.0:
        raise ValueError('score must be between 0.0 and 1.0')
    return f'{score * 100:.1f}%'

class ConfidenceFormattingTests(unittest.TestCase):
    def test_valid_score(self):
        self.assertEqual(format_confidence(0.9123), '91.2%')

    def test_zero_score(self):
        self.assertEqual(format_confidence(0.0), '0.0%')

    def test_invalid_score_raises(self):
        with self.assertRaises(ValueError):
            format_confidence(1.5)

suite = unittest.defaultTestLoader.loadTestsFromTestCase(ConfidenceFormattingTests)
result = unittest.TextTestRunner(verbosity=2).run(suite)
print('All tests passed:', result.wasSuccessful())


## Cell 2: Measure Inference-Like Latency

Repeated timing lets you estimate the typical performance of a code path.


In [ ]:
import time

measurements_ms = []
for _ in range(8):
    start = time.perf_counter()
    sum(i * i for i in range(20_000))
    end = time.perf_counter()
    measurements_ms.append((end - start) * 1000)

print('Measurements (ms):')
print([round(value, 3) for value in measurements_ms])


## Cell 3: Compute Performance Statistics

A single timing can be misleading. Mean and standard deviation reveal consistency.


In [ ]:
import statistics

mean_ms = statistics.mean(measurements_ms)
stdev_ms = statistics.stdev(measurements_ms) if len(measurements_ms) > 1 else 0.0
fastest = min(measurements_ms)
slowest = max(measurements_ms)

print(f'Mean latency:   {mean_ms:.3f} ms')
print(f'Std deviation:  {stdev_ms:.3f} ms')
print(f'Fastest run:    {fastest:.3f} ms')
print(f'Slowest run:    {slowest:.3f} ms')


## Cell 4: Parse Logs for Warnings and Errors

Debugging often starts with log analysis rather than code changes.


In [ ]:
sample_logs = [
    'INFO Camera opened successfully',
    'INFO Image resized to 224x224',
    'WARNING Confidence below threshold',
    'ERROR Network timeout while uploading image',
    'INFO Falling back to offline mode',
]

warnings = [line for line in sample_logs if line.startswith('WARNING')]
errors = [line for line in sample_logs if line.startswith('ERROR')]

print('Warnings:', warnings)
print('Errors:', errors)


## Cell 5: Generate Test Cases and Edge Cases

Strong testing includes normal flows and difficult boundary conditions.


In [ ]:
edge_cases = [
    {'name': 'empty_image_path', 'input': ''},
    {'name': 'oversized_image', 'input': '12000x9000'},
    {'name': 'confidence_negative', 'input': -0.1},
    {'name': 'xml_missing_treatment', 'input': '<disease></disease>'},
    {'name': 'network_unavailable', 'input': None},
]

print('=== EDGE CASE INVENTORY ===')
for index, case in enumerate(edge_cases, start=1):
    print(f'{index}. {case["name"]:<22} -> {case["input"]}')


## Cell 6: Turn Edge Cases into a Simple Coverage Plan

Good test planning links each risky scenario to a test type.


In [ ]:
coverage_plan = {
    'unit_tests': ['formatting helpers', 'input validators', 'XML parsers'],
    'integration_tests': ['camera to result flow', 'API upload flow', 'database save flow'],
    'manual_tests': ['permission denial', 'slow network', 'rotation / lifecycle changes'],
}

for test_type, topics in coverage_plan.items():
    print(f'{test_type}:')
    for topic in topics:
        print(f'  - {topic}')


## Summary and Quick Quiz

**Key takeaways:**
- Unit tests protect small logic units from regression.
- Performance data should be summarized, not guessed.
- Logs and edge-case lists help turn vague bugs into actionable tasks.

**Quick quiz:**
1. Why is standard deviation useful when measuring latency?
2. What is the difference between a unit test and an integration test?
3. Why should edge cases be written down explicitly?
4. How can logs shorten debugging time?


## Parallel Kotlin Track — Week 11: Testing & Debugging in Kotlin

The testing pyramid, debugging workflow, and performance analysis above apply to both tracks.
The Kotlin module declares the **same test dependencies** (JUnit 4, Mockito, Espresso,
androidx.test) as the Java module.

### Java vs Kotlin: a unit test

Java:

```java
@Test
public void confidencePercentage_isScaledBy100() {
    assertEquals(72, Math.round(0.72f * 100f));
}
```

Kotlin:

```kotlin
@Test
fun `confidence percentage is scaled by 100`() {
    assertEquals(72, (0.72f * 100f).roundToInt())
}
```

Kotlin allows backtick-quoted function names, making test reports read like specifications.

### Testing coroutine-based DB code

The Kotlin track replaced `ExecutorService` with `suspend` DAO functions, so tests use
`kotlinx-coroutines-test`'s `runTest { }` (or simply `runBlocking { }` for instrumented tests):

```kotlin
@Test
fun insertedScan_isReturnedNewestFirst() = runBlocking {
    dao.insertScan(ScanRecord(diseaseName = "Tomato Early Blight", timestamp = 2L))
    dao.insertScan(ScanRecord(diseaseName = "Apple Scab", timestamp = 1L))
    val scans = dao.getAllScans()
    assertEquals("Tomato Early Blight", scans.first().diseaseName)
}
```

### Debugging tips specific to the Kotlin track
- `NullPointerException` mostly becomes a **compile-time** error — trust the compiler warnings.
- A crash "cannot find implementation for AppDatabase" means `kapt` is missing (see Week 07 section).
- Coroutine stack traces show `invokeSuspend` frames; enable `-Dkotlinx.coroutines.debug` locally for readable traces.

**✅ Checkpoint:** run `./gradlew test` in `android-app-kotlin/` and confirm the same green result as the Java track.

**⚠️ Common mistake:** forgetting that Espresso UI tests are unchanged — the XML ids are shared, so the same `onView(withId(...))` matchers work in both tracks.

**🔑 Key point:** identical layouts + identical behavior means one set of UI test cases validates both tracks.